In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Coisas a serem feitas:

- [ ] Validar se: A implementação deve ser generalizada para qualquer quantidade de atributos.
- [ ] **Testes Unitários:** validar individualmente métodos como `fit()`, `predict()`, funções de erro, entre outros.
- [ ] **Testes Funcionais:** validar o pipeline completo de treinamento e predição com conjuntos de dados simples.

---

### [ ] Validar: Toy Dataset
* [x] * Criar um pequeno conjunto de dados artificial com cerca de 4 exemplos e 1 atributo.
* [x] O objetivo é validar se o algoritmo encontra uma solução correta com erro na ordem de $10^{-3}$.
* []**Exemplo:** $y = 2x$.

---

### [ ] Validar: Dataset Real
* Utilizar um dataset público voltado para regressão disponível no [UCI Machine Learning Repository](https://archive.ics.uci.edu/) ou no [Kaggle](https://www.kaggle.com/).

---

### [ ] Validar: Divisão dos Dados
* Separar o dataset em três conjuntos: **treinamento**, **desenvolvimento** e **teste**.

#### Calibração da Taxa de Aprendizado:
* Utilizar o conjunto de desenvolvimento para encontrar a melhor taxa de aprendizado (*learning rate*).
* Após encontrar o melhor valor, retreinar o modelo utilizando o conjunto de **treinamento + desenvolvimento**, e avaliar no conjunto de **teste**.

#### Análises Obrigatórias:
* **Curva de Treinamento:**
  * Gerar um gráfico que mostre a evolução da função de erro durante o treinamento.
  * Realizar uma análise textual sobre o comportamento da curva de erro.
* **Análise dos Parâmetros:**
  * Criar um gráfico de barras exibindo os coeficientes encontrados.
  * Analisar em texto quais parâmetros são mais relevantes positiva e negativamente para a predição.

#### Avaliação Quantitativa:
* Calcular e reportar as métricas:
  * **MAE** (*Mean Absolute Error*)
  * **MSE** (*Mean Squared Error*)
  * **MAPE** (*Mean Absolute Percentage Error*)
* As métricas devem ser implementadas em módulos ou classes próprias.
* Comparar os erros de treinamento+desenvolvimento e teste.
* Analisar se há *overfitting* ou *underfitting*, justificando os resultados.

#### Análise de Tempo de Treinamento:
* Medir o tempo total de treinamento.
* Comentar sobre o desempenho temporal observado.

In [2]:
# Gerando os dados sintéticos, função y = x^2

df_sintetico = pd.DataFrame({
                'x': range(50),
                'y': [i*2 for i in range(50)]
})

In [3]:
df_sintetico.head(5)

,x,y
0,0,0
1,1,2
2,2,4
3,3,6
4,4,8


# Fórmula dos gradiente dos mínimos quadradados para múltiplos atributos:

- Função de custo
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^m (h_\theta(x^{(i)}) - y_i)^2$$

- Gradiente descedente para múltiplos atributos:

$$\theta _j := \theta _j - \alpha \frac{\partial}{\partial \theta _j} J(\theta)$$

# Como funciona

$\theta ^T$ = $\begin{bmatrix}\theta _0 & \theta _1 & \theta_2 & ... & \theta _n \end{bmatrix}_{1xn}$ 

x = $\begin{bmatrix} x_0\\ x_1  \\ ...  \\ x_n \end{bmatrix}_{nx1}$

Ou seja, cada linha corresponde a uma feature.

Onde a quantidade de colunas de j determinarão a quantidade de "predições" feitas pelo modelo, dado que a multiplicação de $\theta ^T$ * x vai ser dada por:

$ \theta ^T * x $ = $ \theta _0 * x_0 + \theta _1 * x_1 + ... + \theta _n * x_n $

- Com as dimensões de x sendo $x_{nx2}$, teríamos algo como:

$ \theta ^T * x $ = $\begin{bmatrix} \theta _0 * x_0^0 + \theta _1 * x_1^0 + ... + \theta _n * x_n^0 & \theta _0 * x_0^1 + \theta _1 * x_1^1 + ... + \theta _n * x_n^1  \end{bmatrix}_{1x2}$


In [ ]:
class RTrainer:
    # Etapa de treinamento, é a mais simples, implementação está no slide
    def __init__(self, a, ephocs, erro=0, logger=False):
        """Construtor da classe RTrainer"""
            
        self.a = a # Taxa de aprendizado
        self.ephocs = ephocs # Quantidade de iterações do modelo
        self.erro = erro # Erro mínimo, um parâmetro adicional na hora de iterar, opcional
        self.logger = logger # Um log para acompanharmos algumas coisas que possam ser interessantes de serem observadas
        self.parametros: np.ndarray

    def fit(self, dados_entrada: np.ndarray, y: np.ndarray) -> np.ndarray:
        """ Método que realiza o treinamento do modelo """
    
        # Antes de começar o treinamento, vamos primeiro modelar os dados de forma a facilitar o próprio

        x = np.append(
                [np.ones((dados_entrada).shape[0])], # O shape é para preencher com a quantidade de dados de entrada que temos nas outras features, esse aqui é o x0 -> vetor de 1s
                # TODO: Validar, precisei adicionar '[]' porque só temos uma coluna de entrada
                [np.transpose(dados_entrada)],  # Necessário transpor porque cada linha fica como uma feature
                axis=0 # Garantindo que vamos manter uma estrutura de: cada linha é uma feature, e cada coluna é uma entrada que resulta nos dados de treinamento, que acaba sendo o contrário do que temos no slide
            )
            
        # Ok, temos a matriz dos dados de entrada, agora precisamos criar os parâmetros

        self.parametros = np.ones(x.shape[0]) # TODO: adicionar um parâmetro opcional que possa definir o tipo de inicialização desse vetor de parâmetros, se é rand, 0 e 1, vai ser um if né simples
        parametros_temp = self.parametros.copy()

        # Agora podemos realizar as predições
        for i in range(self.ephocs+1):

            # Primeiro, precisamos calcular o vetor y_pred, que vai ser uma multiplicação de matrizes

            y_pred = self.parametros @ x # O "@" é o operador de multiplicação de matrizes do numpy
          
            # Segundo, calculando a função de custo

            custo = 1/(2*len(x[0])) * np.sum((y_pred - y)**2) # Só se quisermos ver o comportamento do custo, mas não é necessário essa linha aqui

            # Solução com erro 10-3

            if custo < 0.001:
                # print("quebrado!") -> loggando
                break

            # Terceiro, aplicando o método do gradiente descendente para múltiplos atributos

            for j in range(x.shape[0]): 
                # Iterando por feature
                parametros_temp[j] = self.parametros[j] - self.a * (1/(len(x[0]))) * np.sum((y_pred - y) * x[j])

            self.parametros = parametros_temp

        return self.parametros


    
    def predict(self, dados_predicao: np.ndarray) -> float:
        """Método que realiza previsão do modelo, vai retornar -> um vetor de parâmetros OU um objeto MODEL contendo ao menos uma propriedade que indique o número de parâmetros"""
        # TODO: o usuário não vai passar os dois valores, 1 e o outro né, ver de corrigir aqui dentro        
        return  self.parametros @ np.transpose(dados_predicao)
        

In [11]:
testando = RTrainer(
    a = 0.0001,
    ephocs=100000,
    logger=True
)

vetores_peso = testando.fit(df_sintetico['x'].to_numpy(), df_sintetico['y'].to_numpy())

quebrado!


In [12]:
testando.predict(np.array([1, 100]))

np.float64(199.82101089577915)